# Fraud Detection — EDA & Modeling
End-to-end exploration, training, and explainability for the synthetic transactions dataset.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../ml'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
sns.set_theme(style='whitegrid')
DATA = Path('../data/transactions.csv')
if not DATA.exists():
    DATA.parent.mkdir(parents=True, exist_ok=True)
    from generate_data import generate_transactions
    generate_transactions().to_csv(DATA, index=False)
df = pd.read_csv(DATA, parse_dates=['timestamp'])
df.head()

## 1. Class balance & summary stats

In [ ]:
print(df.shape)
print(f'Fraud rate: {df.is_fraud.mean():.3%}')
df.describe()

## 2. Distributions by class

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, ['amount', 'time_of_day', 'user_velocity', 'geo_distance']):
    for label, sub in df.groupby('is_fraud'):
        ax.hist(np.log1p(sub[col]) if col == 'amount' else sub[col], bins=50, alpha=0.5, label=f'fraud={label}')
    ax.set_title(col); ax.legend()
plt.tight_layout()

## 3. Merchant category fraud rate

In [ ]:
rate = df.groupby('merchant_category').is_fraud.mean().sort_values(ascending=False)
rate.plot.bar(figsize=(10, 4), title='Fraud rate by merchant category')
plt.ylabel('Fraud rate')

## 4. Train models

In [ ]:
from train import main as train_main
train_main(n_trials=8)

## 5. SHAP explainability

In [ ]:
from explain import main as explain_main
explain_main()
from IPython.display import Image
Image('../models/shap_xgboost.png')

## 6. Model comparison

In [ ]:
import json
with open('../models/metrics.json') as f:
    m = json.load(f)
pd.DataFrame(m).T[['roc_auc', 'pr_auc']].plot.bar(figsize=(8, 4), title='Model performance')
plt.ylim(0, 1)